# 🛡️ Advanced Email Spam Classification
## Professional ML Pipeline with Advanced Visualizations
**Arch Technologies Internship - Task 1**

---

In [ ]:
# Cell 1: Upload Dataset
from google.colab import files
import pandas as pd

print("📁 Upload emails.csv:")
uploaded = files.upload()

df = pd.read_csv('emails.csv')
print("✅ Dataset Loaded!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Cell 2: Library Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
import warnings
warnings.filterwarnings('ignore')

# For advanced visualizations
from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix
from sklearn.metrics import classification_report, roc_auc_score
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.facecolor'] = 'white'

print("✅ All libraries imported!")

In [ ]:
# Cell 3: Data Cleaning & Feature Engineering
df.columns = df.columns.str.strip().str.lower()
if 'spam' in df.columns:
    df = df.rename(columns={'spam': 'label'})

df = df[['label', 'text']].dropna()
if df['label'].dtype == 'object':
    df['label'] = df['label'].str.strip().str.lower().map({'spam': 1, 'ham': 0, '1': 1, '0': 0})

# Feature engineering
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()
df['avg_word_length'] = df['text_length'] / df['word_count']
df['uppercase_ratio'] = df['text'].str.count('[A-Z]') / df['text_length']
df['special_char_ratio'] = df['text'].str.count('[!@#$%^&*]') / df['text_length']
df['digit_ratio'] = df['text'].str.count('[0-9]') / df['text_length']

print("✅ Data cleaned & engineered!")
print(f"Total emails: {len(df)}")
print(f"Features: {df.columns.tolist()}")

## 📊 Section 1: Exploratory Data Analysis (EDA)

In [ ]:
# Cell 4: Basic Statistics
print("\n" + "="*80)
print("📊 COMPREHENSIVE STATISTICS")
print("="*80)

legit = df[df['label'] == 0]
spam = df[df['label'] == 1]

stats_data = {
    'Metric': ['Count', 'Mean Length', 'Median Length', 'Std Dev', 'Min Length', 'Max Length',
               'Mean Word Count', 'Mean Uppercase %', 'Mean Special Char %', 'Mean Digit %'],
    'Legitimate': [
        len(legit),
        f"{legit['text_length'].mean():.2f}",
        f"{legit['text_length'].median():.2f}",
        f"{legit['text_length'].std():.2f}",
        legit['text_length'].min(),
        legit['text_length'].max(),
        f"{legit['word_count'].mean():.2f}",
        f"{legit['uppercase_ratio'].mean()*100:.2f}",
        f"{legit['special_char_ratio'].mean()*100:.2f}",
        f"{legit['digit_ratio'].mean()*100:.2f}"
    ],
    'Spam': [
        len(spam),
        f"{spam['text_length'].mean():.2f}",
        f"{spam['text_length'].median():.2f}",
        f"{spam['text_length'].std():.2f}",
        spam['text_length'].min(),
        spam['text_length'].max(),
        f"{spam['word_count'].mean():.2f}",
        f"{spam['uppercase_ratio'].mean()*100:.2f}",
        f"{spam['special_char_ratio'].mean()*100:.2f}",
        f"{spam['digit_ratio'].mean()*100:.2f}"
    ]
}

stats_df = pd.DataFrame(stats_data)
print(stats_df.to_string(index=False))

# Skewness & Kurtosis
print("\n" + "="*80)
print("📐 DISTRIBUTION ANALYSIS")
print("="*80)
print(f"\nSkewness (Length):")
print(f"  Legitimate: {skew(legit['text_length']):.4f}")
print(f"  Spam: {skew(spam['text_length']):.4f}")
print(f"\nKurtosis (Length):")
print(f"  Legitimate: {kurtosis(legit['text_length']):.4f}")
print(f"  Spam: {kurtosis(spam['text_length']):.4f}")

In [ ]:
# Cell 5: Advanced Visualization 1 - Interactive Class Distribution
fig = px.pie(names=['Legitimate', 'Spam'],
             values=[len(legit), len(spam)],
             color_discrete_map={'Legitimate': '#2ecc71', 'Spam': '#e74c3c'},
             title='📊 Email Class Distribution',
             labels=['Legitimate', 'Spam'])

fig.update_traces(textposition='inside', textinfo='label+value+percent')
fig.show()

print("✅ Pie chart generated")
print(f"Legitimate: {len(legit)} ({len(legit)/len(df)*100:.1f}%)")
print(f"Spam: {len(spam)} ({len(spam)/len(df)*100:.1f}%)")

In [ ]:
# Cell 6: Advanced Visualization 2 - Violin Plots (Distribution Shape)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('📊 Feature Distributions: Legitimate vs Spam', fontsize=16, fontweight='bold')

features = ['text_length', 'word_count', 'avg_word_length', 'uppercase_ratio', 'special_char_ratio', 'digit_ratio']
titles = ['Text Length', 'Word Count', 'Avg Word Length', 'Uppercase %', 'Special Char %', 'Digit %']

for idx, (feature, title) in enumerate(zip(features, titles)):
    ax = axes[idx // 3, idx % 3]
    
    parts = ax.violinplot([legit[feature].values, spam[feature].values],
                           positions=[0, 1],
                           showmeans=True,
                           showmedians=True)
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Legitimate', 'Spam'])
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_ylabel('Value', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add color
    for pc in parts['bodies']:
        pc.set_facecolor('#3498db')
        pc.set_alpha(0.7)

plt.tight_layout()
plt.savefig('06_violin_plots.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Violin plots saved")


In [ ]:
# Cell 7: Advanced Visualization 3 - KDE Plots (Smooth Distribution)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('📊 Kernel Density Estimation (KDE)', fontsize=16, fontweight='bold')

for idx, (feature, title) in enumerate(zip(features, titles)):
    ax = axes[idx // 3, idx % 3]
    
    legit[feature].plot(kind='kde', ax=ax, label='Legitimate', linewidth=2.5, color='#2ecc71')
    spam[feature].plot(kind='kde', ax=ax, label='Spam', linewidth=2.5, color='#e74c3c')
    
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel('Value', fontweight='bold')
    ax.set_ylabel('Density', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.fill_between(ax.get_lines()[0].get_xdata(), ax.get_lines()[0].get_ydata(), alpha=0.2, color='#2ecc71')
    ax.fill_between(ax.get_lines()[1].get_xdata(), ax.get_lines()[1].get_ydata(), alpha=0.2, color='#e74c3c')

plt.tight_layout()
plt.savefig('07_kde_plots.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ KDE plots saved")


In [ ]:
# Cell 8: Advanced Visualization 4 - Correlation Heatmap
fig, ax = plt.subplots(figsize=(12, 9))

feature_cols = ['text_length', 'word_count', 'avg_word_length', 'uppercase_ratio', 'special_char_ratio', 'digit_ratio', 'label']
corr_matrix = df[feature_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.3f',
            square=True, ax=ax, cbar_kws={'label': 'Correlation'},
            linewidths=2, linecolor='white')

ax.set_title('🔥 Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('08_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Correlation heatmap saved")


## ⚙️ Section 2: Feature Extraction & Preprocessing

In [ ]:
# Cell 9: Text Preprocessing & TF-IDF
import re, string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
nltk.download('stopwords')
nltk.download('wordnet')

STOPWORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens if t not in STOPWORDS and len(t) > 2]
    return ' '.join(tokens)

print("🔄 Text Preprocessing...")
df['clean_text'] = df['text'].apply(clean_text)

print("🔄 TF-IDF Vectorization...")
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.8)
X = vectorizer.fit_transform(df['clean_text'])
y = df['label'].values

print(f"✅ Features extracted!")
print(f"   Shape: {X.shape}")
print(f"   Sparsity: {(1 - X.nnz / (X.shape[0] * X.shape[1])) * 100:.2f}%")

# Top features
feature_names = vectorizer.get_feature_names_out()
feature_importance = np.asarray(X.mean(axis=0)).flatten()
top_features_idx = feature_importance.argsort()[-20:][::-1]

print(f"\n📊 TOP 20 FEATURES:")
for i, idx in enumerate(top_features_idx, 1):
    print(f"{i:2d}. {feature_names[idx]:<30} Score: {feature_importance[idx]:.6f}")

In [ ]:
# Cell 10: Visualization 5 - Top Features Bar Chart
fig, ax = plt.subplots(figsize=(12, 8))

top_20_features = feature_names[top_features_idx]
top_20_scores = feature_importance[top_features_idx]

colors = ['#e74c3c' if 'free' in f or 'click' in f or 'urgent' in f else '#2ecc71' for f in top_20_features]
bars = ax.barh(range(len(top_20_features)), top_20_scores, color=colors, edgecolor='black', linewidth=1.5)

ax.set_yticks(range(len(top_20_features)))
ax.set_yticklabels(top_20_features, fontsize=11)
ax.set_xlabel('TF-IDF Score', fontsize=12, fontweight='bold')
ax.set_title('🎯 Top 20 Most Important Features', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('10_top_features.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Top features visualization saved")


## 🤖 Section 3: Model Training & Evaluation

In [ ]:
# Cell 11: Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("\n" + "="*70)
print("📚 TRAIN-TEST SPLIT")
print("="*70)
print(f"\nTotal samples: {len(df)}")
print(f"Training set: {X_train.shape[0]} ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"Testing set: {X_test.shape[0]} ({X_test.shape[0]/len(df)*100:.1f}%)")
print(f"\nSpam ratio in training: {y_train.sum()/len(y_train)*100:.2f}%")
print(f"Spam ratio in testing: {y_test.sum()/len(y_test)*100:.2f}%")


In [ ]:
# Cell 12: Train Multiple Models
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

print("\n" + "="*70)
print("🤖 TRAINING 5 MODELS")
print("="*70)

models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'SVM': SVC(kernel='linear', probability=True, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42),
}

trained_models = {}
for name, model in models.items():
    print(f"\n🚀 Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"   ✅ Done!")

print("\n✅ All models trained successfully!")


In [ ]:
# Cell 13: Comprehensive Model Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve, auc, roc_auc_score, precision_recall_curve

print("\n" + "="*70)
print("📊 MODEL EVALUATION RESULTS")
print("="*70)

results = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    
    # For probability predictions
    if hasattr(model, 'predict_proba'):
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_pred_proba = model.decision_function(X_test)
        y_pred_proba = 1 / (1 + np.exp(-y_pred_proba))
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    cm = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_proba)
    
    results[name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': roc_auc,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'cm': cm,
        'fpr': fpr,
        'tpr': tpr,
        'precision_vals': precision_vals,
        'recall_vals': recall_vals,
        'auc': auc(fpr, tpr)
    }
    
    print(f"\n{'='*50}")
    print(f"🎯 {name}")
    print(f"{'='*50}")
    print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  TN: {cm[0][0]:5d}  FP: {cm[0][1]:5d}")
    print(f"  FN: {cm[1][0]:5d}  TP: {cm[1][1]:5d}")
    print(f"\nSensitivity: {cm[1][1]/(cm[1][1]+cm[1][0]):.4f} (catches spam)")
    print(f"Specificity: {cm[0][0]/(cm[0][0]+cm[0][1]):.4f} (avoids false alarms)")


In [ ]:
# Cell 14: Visualization 6 - Metrics Comparison Bar Chart
fig, ax = plt.subplots(figsize=(14, 7))

metrics_names = list(results.keys())
metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
colors_metrics = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

x = np.arange(len(metrics_names))
width = 0.15

for i, metric in enumerate(metrics):
    values = [results[name][metric] for name in metrics_names]
    ax.bar(x + i*width, values, width, label=metric.upper(), color=colors_metrics[i], edgecolor='black', linewidth=1.5)

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('📊 Model Metrics Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(metrics_names, fontsize=11)
ax.legend(fontsize=10, loc='lower right')
ax.set_ylim([0.85, 1.01])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('14_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Metrics comparison saved")


In [ ]:
# Cell 15: Visualization 7 - ROC Curves for All Models
fig, ax = plt.subplots(figsize=(10, 8))

for name, result in results.items():
    ax.plot(result['fpr'], result['tpr'], linewidth=2.5,
            label=f"{name} (AUC = {result['auc']:.4f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('📈 ROC Curves - All Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('15_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ ROC curves saved")


In [ ]:
# Cell 16: Visualization 8 - Precision-Recall Curves
fig, ax = plt.subplots(figsize=(10, 8))

for name, result in results.items():
    ax.plot(result['recall_vals'], result['precision_vals'], linewidth=2.5,
            label=f"{name} (AUC = {auc(result['recall_vals'], result['precision_vals']):.4f})")

ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
ax.set_title('📈 Precision-Recall Curves - All Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('16_precision_recall_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Precision-Recall curves saved")


In [ ]:
# Cell 17: Visualization 9 - Confusion Matrices (All Models)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('🔍 Confusion Matrices - All Models', fontsize=16, fontweight='bold')

for idx, (name, result) in enumerate(results.items()):
    ax = axes[idx // 3, idx % 3]
    cm = result['cm']
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                cbar=False, annot_kws={'size': 12, 'fontweight': 'bold'},
                xticklabels=['Legitimate', 'Spam'],
                yticklabels=['Legitimate', 'Spam'])
    
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.set_ylabel('Actual', fontweight='bold')
    ax.set_xlabel('Predicted', fontweight='bold')

# Hide the extra subplot
fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.savefig('17_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Confusion matrices saved")


In [ ]:
# Cell 18: Visualization 10 - Interactive Metrics Table
import plotly.graph_objects as go

metric_data = []
for name in results.keys():
    metric_data.append([
        name,
        f"{results[name]['accuracy']:.4f}",
        f"{results[name]['precision']:.4f}",
        f"{results[name]['recall']:.4f}",
        f"{results[name]['f1_score']:.4f}",
        f"{results[name]['roc_auc']:.4f}"
    ])

fig = go.Figure(data=[go.Table(
    header=dict(values=['<b>Model</b>', '<b>Accuracy</b>', '<b>Precision</b>', '<b>Recall</b>', '<b>F1-Score</b>', '<b>ROC-AUC</b>'],
                fill_color='#1f77b4',
                font=dict(color='white', size=12),
                align='center',
                height=30),
    cells=dict(values=list(zip(*metric_data)),
               fill_color='#f0f0f0',
               font=dict(size=11),
               align='center',
               height=25)
)])

fig.update_layout(title_text='📊 Model Performance Summary', title_font_size=16)
fig.show()
print("✅ Interactive metrics table created")


In [ ]:
# Cell 19: Visualization 11 - Best Model Confusion Matrix (Large)
best_name = max(results, key=lambda x: results[x]['f1_score'])
best_cm = results[best_name]['cm']

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(best_cm, annot=True, fmt='d', cmap='RdYlGn_r', ax=ax,
            cbar_kws={'label': 'Count'},
            xticklabels=['Legitimate', 'Spam'],
            yticklabels=['Legitimate', 'Spam'],
            annot_kws={'size': 16, 'fontweight': 'bold'},
            linewidths=3,
            linecolor='black')

ax.set_title(f'🏆 Best Model: {best_name}\nConfusion Matrix (Normalized)', fontsize=14, fontweight='bold')
ax.set_ylabel('Actual Label', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')

# Add metrics text
tn, fp, fn, tp = best_cm[0][0], best_cm[0][1], best_cm[1][0], best_cm[1][1]
accuracy_text = f"Accuracy: {results[best_name]['accuracy']:.4f}\nPrecision: {results[best_name]['precision']:.4f}\nRecall: {results[best_name]['recall']:.4f}\nF1-Score: {results[best_name]['f1_score']:.4f}"
plt.text(2.5, 1, accuracy_text, fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
         fontweight='bold')

plt.tight_layout()
plt.savefig('19_best_model_confusion.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Best model confusion matrix saved: {best_name}")


## 🎯 Section 4: Advanced Analysis

In [ ]:
# Cell 20: Visualization 12 - Model Comparison Radar Chart
from math import pi

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
N = len(categories)

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

for name, result in results.items():
    values = [
        result['accuracy'],
        result['precision'],
        result['recall'],
        result['f1_score'],
        result['roc_auc']
    ]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=name, markersize=6)
    ax.fill(angles, values, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11, fontweight='bold')
ax.set_ylim(0.8, 1.0)
ax.set_yticks([0.82, 0.86, 0.90, 0.94, 0.98])
ax.set_yticklabels(['0.82', '0.86', '0.90', '0.94', '0.98'], fontsize=9)
ax.grid(True)
ax.set_title('🎯 Multi-Metric Model Comparison (Radar)', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

plt.tight_layout()
plt.savefig('20_radar_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Radar chart created")


In [ ]:
# Cell 21: Visualization 13 - Learning Curves (for best model)
from sklearn.model_selection import learning_curve

best_model = trained_models[best_name]

train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train, cv=5, 
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1, scoring='f1'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(train_sizes, train_mean, 'o-', linewidth=2, label='Training Score', color='#2ecc71', markersize=8)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color='#2ecc71')

ax.plot(train_sizes, val_mean, 'o-', linewidth=2, label='Validation Score', color='#e74c3c', markersize=8)
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2, color='#e74c3c')

ax.set_xlabel('Training Set Size', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title(f'📈 Learning Curves - {best_name}', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.8, 1.02])

plt.tight_layout()
plt.savefig('21_learning_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Learning curves saved")


In [ ]:
# Cell 22: Final Summary Report
print("\n" + "="*80)
print("🏆 FINAL SUMMARY & RECOMMENDATIONS")
print("="*80)

print(f"\n🎯 BEST PERFORMING MODEL: {best_name}")
print(f"\n{'Metric':<20} {'Value':<15}")
print("-"*35)
for metric in ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']:
    print(f"{metric.replace('_', ' ').title():<20} {results[best_name][metric]:<15.4f}")

print(f"\n📊 CONFUSION MATRIX BREAKDOWN:")
best_cm = results[best_name]['cm']
print(f"  True Negatives:  {best_cm[0][0]}")
print(f"  False Positives: {best_cm[0][1]}")
print(f"  False Negatives: {best_cm[1][0]}")
print(f"  True Positives:  {best_cm[1][1]}")

print(f"\n💡 KEY INSIGHTS:")
print(f"  • Model catches {(best_cm[1][1]/(best_cm[1][1]+best_cm[1][0])*100):.1f}% of spam emails")
print(f"  • False alarm rate: {(best_cm[0][1]/(best_cm[0][0]+best_cm[0][1])*100):.1f}%")
print(f"  • Correctly identifies {(best_cm[0][0]/(best_cm[0][0]+best_cm[0][1])*100):.1f}% of legitimate emails")

print(f"\n✅ ALL VISUALIZATIONS GENERATED:")
print(f"  06_violin_plots.png")
print(f"  07_kde_plots.png")
print(f"  08_correlation_heatmap.png")
print(f"  10_top_features.png")
print(f"  14_metrics_comparison.png")
print(f"  15_roc_curves.png")
print(f"  16_precision_recall_curves.png")
print(f"  17_confusion_matrices.png")
print(f"  19_best_model_confusion.png")
print(f"  20_radar_chart.png")
print(f"  21_learning_curves.png")

print("\n" + "="*80)


In [ ]:
# Cell 23: Save All Results to JSON
import json

summary = {
    'best_model': best_name,
    'models': {
        name: {
            'accuracy': float(result['accuracy']),
            'precision': float(result['precision']),
            'recall': float(result['recall']),
            'f1_score': float(result['f1_score']),
            'roc_auc': float(result['roc_auc'])
        }
        for name, result in results.items()
    },
    'dataset': {
        'total_emails': len(df),
        'legitimate': len(legit),
        'spam': len(spam),
        'spam_percentage': float(len(spam)/len(df)*100)
    }
}

with open('results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("✅ Results saved to results_summary.json")
print(json.dumps(summary, indent=2))
